# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata and initialize mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)
# Metadata is an mlcroissant.DatasetMetadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Explore record sets and fields using their `@id`s
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    print("Fields:")
    for field in rs.fields:
        print(f"  Field name: {field.name}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
    print('-'*40)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set into dataframes
dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    print(f"Loading records from record set: {rs.name} (@id: {rs_id})")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns for {rs_id}: {df.columns.tolist()}")
    print(df.head(), "\n---\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section shows how to remove outliers, transform data distributions, or group data by key attributes to prepare it for further analysis.

Below, we select a numeric field from the survey record set and perform filtering, normalization, and grouping.

In [ ]:
# Choose representative record set and fields for analysis
# Find the main survey record set (by name or fields)
survey_rs = None
for rs in record_sets:
    # Try to use heuristics - pick the record set with fields like 'GAD-7', 'PHQ-9', or similar
    survey_field_names = [field.name.lower() for field in rs.fields]
    if any(x in survey_field_names for x in ['gad-7', 'gad7', 'phq-9', 'phq9', 'psq', 'score']):
        survey_rs = rs
        break

if survey_rs is None:
    # fallback: use the first record set
    survey_rs = record_sets[0]

df = dataframes[survey_rs.id]
print(f"Selected record set {survey_rs.name} (@id: {survey_rs.id}) for EDA")

# Find numeric fields: heuristically, those with Float/Integer and names like score or age
numeric_fields = [field for field in survey_rs.fields if getattr(field, 'data_type', '').lower() in ['integer', 'float', 'number']]
if not numeric_fields:
    # fallback: pick columns with object type int/float in df
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) and col != 'id']
    numeric_field_id = numeric_fields[0] if numeric_fields else None
else:
    numeric_field_id = numeric_fields[0].id

print(f"Candidate numeric field for analysis: {numeric_field_id}")

# Filtering example
threshold = 10  # choose threshold for illustration
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by a categorical field
    # Choose a group field, e.g., 'gender' or similar
    group_field_candidates = [field.id for field in survey_rs.fields if getattr(field, 'data_type', '').lower() in ['text', 'string']]
    group_field = group_field_candidates[0] if group_field_candidates else None

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No suitable numeric field for demonstration found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, we plot the distribution of a sample numeric field and (if available) a histogram grouped by a category.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, show grouped means
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
This notebook has demonstrated loading, processing, and visualizing a FAIR²-compliant mental health survey dataset using the `mlcroissant` library. The dataset structure (record sets, fields) makes it easy to reference data entities by their `@id`, enabling reproducible and modular exploration workflows.

Key findings and observations:
- The data contains multiple record sets and various fields including demographic and mental health measurements.
- Numeric fields such as scores enable quantitative analysis, while categorical fields like gender or education can be used for segmentation.
- Filtering and normalization can be readily performed and visualized for further statistical analyses and machine learning tasks.

Use the record set and field `@id`s throughout your analysis for full compatibility with the Croissant data model and future dataset iterations.